In [7]:
import os
import json
import pandas as pd
import re

def analyze_sweeps(root_dir):
    results = []
    
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            # 排除 checkpoint 文件，只处理 jsonl
            if file.endswith(".jsonl") and "-checkpoint" not in file:
                file_path = os.path.join(root, file)
                
                # 提取任务名称 (假设在路径的第三层)
                path_parts = root.split(os.sep)
                task = path_parts[2] if len(path_parts) > 2 else "Unknown"
                
                # 正则匹配新格式: results_layer_12_alpha_0.5
                layer_match = re.search(r'layer_(\d+)', file)
                alpha_match = re.search(r'alpha_([\d\.]+)', file)
                
                layer = layer_match.group(1) if layer_match else "N/A"
                alpha = alpha_match.group(1) if alpha_match else "N/A"
                
                try:
                    correct_count = 0
                    total_count = 0
                    with open(file_path, 'r', encoding='utf-8') as f:
                        for line in f:
                            data = json.loads(line)
                            total_count += 1
                            if data.get("is_correct") is True:
                                correct_count += 1
                    
                    accuracy = (correct_count / total_count) * 100 if total_count > 0 else 0
                    
                    results.append({
                        "Task": task,
                        "Layer": layer,
                        "Alpha": alpha,
                        "Accuracy": accuracy,
                        "File": file
                    })
                except Exception as e:
                    print(f"Error processing {file_path}: {e}")

    df = pd.DataFrame(results)
    if df.empty:
        return None

    # --- 核心修改：将字符串转为数值，以便进行逻辑排序 ---
    df['Layer'] = pd.to_numeric(df['Layer'], errors='coerce')
    df['Alpha'] = pd.to_numeric(df['Alpha'], errors='coerce')

    # 按 Task 分组，然后按 Layer 和 Alpha 升序排列，方便对比
    df = df.sort_values(by=["Task", "Layer", "Alpha"], ascending=[True, True, True])
    
    return df

# 执行分析
root_path = "./static/Qwen2.5-3B-Instruct/LogicalDeduction" 
all_results_df = analyze_sweeps(root_path)

if all_results_df is not None:
    print(f"### 找到共 {len(all_results_df)} 条实验记录 ###")
    
    # 使用 pd.option_context 确保打印所有行，不会因为太长被折叠 (Optional)
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
        print(all_results_df.to_string(index=False)) # 不打印索引号，更整洁
        
    # 如果你想顺便存个 CSV 方便在 Excel 看：
    # all_results_df.to_csv("all_experiment_results.csv", index=False)
else:
    print("未找到有效的结果文件。")

### 找到共 65 条实验记录 ###
               Task  Layer  Alpha  Accuracy                                      File
Qwen2.5-3B-Instruct    2.0    NaN 47.333333  instance_results_layer_2_alpha_0.5.jsonl
Qwen2.5-3B-Instruct    2.0    NaN 51.000000  instance_results_layer_2_alpha_1.0.jsonl
Qwen2.5-3B-Instruct    2.0    NaN 49.333333  instance_results_layer_2_alpha_1.5.jsonl
Qwen2.5-3B-Instruct    4.0    NaN 50.333333  instance_results_layer_4_alpha_0.5.jsonl
Qwen2.5-3B-Instruct    4.0    NaN 50.333333  instance_results_layer_4_alpha_1.0.jsonl
Qwen2.5-3B-Instruct    4.0    NaN 50.666667  instance_results_layer_4_alpha_1.5.jsonl
Qwen2.5-3B-Instruct    6.0    NaN 49.333333  instance_results_layer_6_alpha_0.5.jsonl
Qwen2.5-3B-Instruct    6.0    NaN 50.333333  instance_results_layer_6_alpha_1.0.jsonl
Qwen2.5-3B-Instruct    6.0    NaN 50.333333  instance_results_layer_6_alpha_1.5.jsonl
Qwen2.5-3B-Instruct    8.0    NaN 46.666667  instance_results_layer_8_alpha_0.5.jsonl
Qwen2.5-3B-Instruct    8.0    NaN

In [8]:
# 执行分析
root_path = "./static/Qwen2.5-7B-Instruct/LogicalDeduction" 
all_results_df = analyze_sweeps(root_path)

if all_results_df is not None:
    print(f"### 找到共 {len(all_results_df)} 条实验记录 ###")
    
    # 使用 pd.option_context 确保打印所有行，不会因为太长被折叠 (Optional)
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
        print(all_results_df.to_string(index=False)) # 不打印索引号，更整洁
        
    # 如果你想顺便存个 CSV 方便在 Excel 看：
    # all_results_df.to_csv("all_experiment_results.csv", index=False)
else:
    print("未找到有效的结果文件。")

### 找到共 76 条实验记录 ###
               Task  Layer  Alpha  Accuracy                                      File
Qwen2.5-7B-Instruct    2.0    NaN 68.000000  instance_results_layer_2_alpha_0.5.jsonl
Qwen2.5-7B-Instruct    2.0    NaN 67.666667  instance_results_layer_2_alpha_1.0.jsonl
Qwen2.5-7B-Instruct    2.0    NaN 72.666667  instance_results_layer_2_alpha_1.5.jsonl
Qwen2.5-7B-Instruct    4.0    NaN 70.333333  instance_results_layer_4_alpha_0.5.jsonl
Qwen2.5-7B-Instruct    4.0    NaN 66.333333  instance_results_layer_4_alpha_1.0.jsonl
Qwen2.5-7B-Instruct    4.0    NaN 70.333333  instance_results_layer_4_alpha_1.5.jsonl
Qwen2.5-7B-Instruct    6.0    NaN 71.333333  instance_results_layer_6_alpha_0.5.jsonl
Qwen2.5-7B-Instruct    6.0    NaN 69.000000  instance_results_layer_6_alpha_1.0.jsonl
Qwen2.5-7B-Instruct    6.0    NaN 69.000000  instance_results_layer_6_alpha_1.5.jsonl
Qwen2.5-7B-Instruct    8.0    NaN 66.666667  instance_results_layer_8_alpha_0.5.jsonl
Qwen2.5-7B-Instruct    8.0    NaN

In [ ]:
# 执行分析
root_path = "./static/Qwen2.5-7B-Instruct/FOLIO" 
all_results_df = analyze_sweeps(root_path)

if all_results_df is not None:
    print(f"### 找到共 {len(all_results_df)} 条实验记录 ###")
    
    # 使用 pd.option_context 确保打印所有行，不会因为太长被折叠 (Optional)
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
        print(all_results_df.to_string(index=False)) # 不打印索引号，更整洁
        
    # 如果你想顺便存个 CSV 方便在 Excel 看：
    # all_results_df.to_csv("all_experiment_results.csv", index=False)
else:
    print("未找到有效的结果文件。")

### 找到共 4 条实验记录 ###
               Task  Layer  Alpha  Accuracy                                     File
Qwen2.5-3B-Instruct   12.0    NaN  0.000000         results_layer_12_alpha_1.0.jsonl
Qwen2.5-3B-Instruct   12.0    NaN 55.882353         results_layer_12_alpha_0.5.jsonl
Qwen2.5-3B-Instruct    NaN    NaN 60.294118         results_baseline_alpha_0.0.jsonl
Qwen2.5-3B-Instruct    NaN    NaN 59.313725 results_reverse_baseline_alpha_0.0.jsonl
